# Convert client_idcodes (hospital numbers) to NHS numbers using epr_documents index

For legacy data in the epr_documents index:
- `client_idcode` = Hospital number
- `client_universalnumber` = NHS number

In [ ]:
import pandas as pd

# Import the conversion functions
from pat2vec.util.patient_identifier_conversion import (
    convert_client_idcode_to_nhs_number,
    convert_nhs_number_to_client_idcode,
)

In [ ]:
# Load treatment_docs file (contains client_idcodes which are hospital numbers)
treatment_docs_path = "test_files/treatment_docs.csv"
df = pd.read_csv(treatment_docs_path)

print("Treatment docs structure:")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 3 rows:\n{df.head(3)}")

In [ ]:
# Extract unique client_idcodes (hospital numbers)
client_idcodes = df["client_idcode"].dropna().astype(str).unique().tolist()
print(f"Found {len(client_idcodes)} unique client_idcodes (hospital numbers)")

# Initialize pat2vec object
import pat2vec
from pat2vec.main_pat2vec import main
from pat2vec.util.config_pat2vec import config_class

config_obj = config_class(testing=False)  # Set to False for production
pat2vec_obj = main(config_obj=config_obj, cogstack=True)

In [ ]:
# Convert hospital numbers (client_idcodes) to NHS numbers
print("=== Converting client_idcodes to NHS numbers ===")
nhs_numbers, missing = convert_client_idcode_to_nhs_number(
    client_idcodes=client_idcodes,
    pat2vec_obj=pat2vec_obj,
)

In [ ]:
print(f"\n=== Results ===")
print(f"Hospital numbers processed: {len(client_idcodes)}")
print(f"NHS numbers found: {len(nhs_numbers)}")

if nhs_numbers:
    print("\nNHS Numbers:")
    for i, nhs in enumerate(nhs_numbers, 1):
        print(f"  {i}. {nhs}")

if missing:
    print(f"\nMissing/failure ({len(missing)}):")
    for code in missing:
        print(f"  - {code}")

# Reverse conversion: NHS numbers to client_idcodes (hospital numbers)

In [ ]:
# Convert NHS numbers back to hospital numbers
if nhs_numbers:
    print("=== Converting NHS numbers to client_idcodes ===")
    client_idcodes_from_nhs, missing_nhs = convert_nhs_number_to_client_idcode(
        nhs_numbers=nhs_numbers[:5],  # Test with first 5
        pat2vec_obj=pat2vec_obj,
    )

    print(f"\nNHS numbers processed: {len(nhs_numbers[:5])}")
    print(f"Hospital numbers found: {len(client_idcodes_from_nhs)}")